In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))

import pandas as pd
from regression_modelling.constants import FEATURE_SOURCES
from regression_modelling.data_wrangling import sources, features
from regression_modelling.bias_testing.data import build_protected_table
from crime_blockgroup_mapping.config import INTERIM_DIR
from crime_blockgroup_mapping.constants import UCR_YEAR

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# What sources are declared, and how each is fetched
for name, src in FEATURE_SOURCES.items():
    print(f"{name:12} backend={src.backend:4} location={src.location:10} "
          f"key={src.key_col:6} cols={src.feature_cols}")

vacancy      backend=bq   location=vacancy    key=geoid  cols=('vacant_pct',)
liens        backend=bq   location=liens      key=geoid  cols=('clip_liens_pct',)
foreclosures backend=bq   location=foreclosures key=geoid  cols=('clip_foreclosure_pct',)
convenience_stores backend=bq   location=convenience_stores key=geoid  cols=('unq_convenience_stores_clips',)
gas_stations backend=bq   location=gas_stations key=geoid  cols=('unq_gas_stations_clips',)
liquor_stores backend=bq   location=liquor_stores key=geoid  cols=('unq_liquor_stores_clips',)
transit      backend=file location=build via regression_modelling.data_wrangling.transit.build_all_transit key=geoid  cols=('transit_stop_count', 'transit_stop_density', 'transit_nearest_stop_m', 'transit_service_intensity', 'transit_overnight_stop_count', 'transit_overnight_stop_share', 'transit_risky_stop_count', 'transit_risky_stop_share', 'transit_risky_allnight_count', 'transit_route_mode_diversity')
imagery      backend=bq   location=imagery  

In [ ]:
# --- Build BQ staging tables (run only when missing or upstream data changed) ---
# Property features build from tables reachable here:
#for name in ["vacancy", "liens", "foreclosures"]:
#    sources.run_bq_build(name)
#    print(f"built {name}")

# Store POIs use the generic sql/build/stores.sql template + STORE_DEFS predicates.
# NOTE: firmographics live in a prd project unreachable from this VM, so run the
# rendered DDL in the BQ console (already done for the 3 stores pulled below):
#for store in ["convenience_stores", "gas_stations", "liquor_stores"]:
#    sources.run_bq_build_store(store)   # -> bg_{store}
#    print(f"built bg_{store}")

In [3]:
# Uncached single pull to confirm BQ auth + schema before running the full pipeline
for source in ["vacancy",
               "liens",
               "foreclosures",
               "gas_stations",
               "liquor_stores",
               "convenience_stores",
               "imagery"]:
    df = sources.run_bq_pull(f"{source}")
    print(f"{source} pull: {df.shape}")

/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


vacancy pull: (241456, 4)
liens pull: (241456, 4)
foreclosures pull: (241456, 4)
gas_stations pull: (241456, 4)
liquor_stores pull: (241456, 4)
convenience_stores pull: (241456, 4)
imagery pull: (241362, 7)


In [4]:
# Load liens, vacancy and other parquets saved already
# Check all sources have a geoid column
for name, src in FEATURE_SOURCES.items():
    df = sources.pull_source(src, refresh=False)
    print(f"{name:12} {df.shape}  cols={list(df.columns)}")

vacancy      (241456, 4)  cols=['geoid', 'vacant_pct', 'vacant_addr', 'total_addr']
liens        (241456, 4)  cols=['geoid', 'clip_liens_pct', 'total_clips', 'clip_w_liens']
foreclosures (241456, 4)  cols=['geoid', 'clip_foreclosure_pct', 'total_unq_clips', 'unq_clip_w_foreclosure']
convenience_stores (241456, 4)  cols=['geoid', 'convenience_stores_clip_pct', 'total_unq_clips', 'unq_convenience_stores_clips']
gas_stations (241456, 4)  cols=['geoid', 'gas_stations_clip_pct', 'total_unq_clips', 'unq_gas_stations_clips']
liquor_stores (241456, 4)  cols=['geoid', 'liquor_stores_clip_pct', 'total_unq_clips', 'unq_liquor_stores_clips']
transit      (7863, 12)  cols=['geoid', 'city', 'transit_stop_count', 'transit_stop_density', 'transit_nearest_stop_m', 'transit_service_intensity', 'transit_overnight_stop_count', 'transit_overnight_stop_share', 'transit_risky_stop_count', 'transit_risky_stop_share', 'transit_risky_allnight_count', 'transit_route_mode_diversity']


/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


imagery      (241362, 7)  cols=['geoid', 'roof_condition_avg', 'roof_debris_pct_avg', 'roof_discoloration_pct_avg', 'hardscapes_pct_avg', 'roof_missing_material_pct', 'imagery_structure_count']


In [5]:
# Load demographic features
demo = features.build_demographic_features(refresh=True)
print("demographic:", demo.shape)
demo.head()

demographic: (242335, 10)


,geoid,det_pct,in_household_pct,moved1yr_pct,own_pct,lap_pct,Division,city_centers_dist,pop_est_5mile,pop_ch_1mile
0,020130001001,73.279352,65.293602,16.042465,28.089888,0.404858,9.0,50.0,2066.0,-37.826087
1,020130001002,57.459677,65.293602,16.042465,54.318182,13.709677,9.0,50.0,884.0,-11.588785
2,020130001003,75.638051,65.293602,16.042465,73.684211,10.440835,9.0,50.0,674.0,-32.747604
3,020160001001,33.962264,66.310160,19.562244,39.766082,3.537736,9.0,50.0,1023.0,-6.250000
4,020160002001,9.478673,68.261851,24.745302,21.652422,49.052133,9.0,50.0,4411.0,-0.500835


In [6]:
# Assemble features
# NOTE: set refresh to True whenever a new feature is added
feats = features.assemble_features(refresh=True)
print("bg_predictors:", feats.shape)
feats.info()

/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuer

bg_predictors: (242335, 32)
<class 'pandas.DataFrame'>
RangeIndex: 242335 entries, 0 to 242334
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   geoid                         242335 non-null  str    
 1   det_pct                       242335 non-null  float64
 2   in_household_pct              242335 non-null  float64
 3   moved1yr_pct                  242335 non-null  float64
 4   own_pct                       242335 non-null  float64
 5   lap_pct                       242335 non-null  float64
 6   Division                      242335 non-null  float64
 7   city_centers_dist             239780 non-null  float64
 8   pop_est_5mile                 239392 non-null  float64
 9   pop_ch_1mile                  239392 non-null  float64
 10  vacant_pct                    240235 non-null  float64
 11  clip_liens_pct                241273 non-null  float64
 12  clip_foreclosure_pct       

In [10]:
# Build bias testing table
# NOTE: set refresh to True whenever a new feature is added
bias_feats = build_protected_table(refresh=True)
print("bg_bias_predictors:", bias_feats.shape)
bias_feats.info()

protected table (242335, 3) → /home/eprashar_solutions_corelogic_com/crime-idx-2026/data/interim/bias/protected_attributes.parquet
bg_bias_predictors: (242335, 3)
<class 'pandas.DataFrame'>
RangeIndex: 242335 entries, 0 to 242334
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   geoid     242335 non-null  str    
 1   md_hhinc  224390 non-null  float64
 2   wht_pct   242335 non-null  float64
dtypes: float64(2), str(1)
memory usage: 8.3 MB


In [7]:
# geoid should be a 12-char string, unique per row
print("geoid dtype :", feats["geoid"].dtype)
print("geoid lengths:", feats["geoid"].str.len().value_counts().to_dict())
print("duplicate geoids:", feats["geoid"].duplicated().sum())

# Non-null coverage per column — low coverage on vacancy/liens flags join or fill issues
print("\nNon-null coverage:")
print((feats.notna().mean() * 100).round(1).astype(str) + "%")

geoid dtype : str
geoid lengths: {12: 242335}
duplicate geoids: 0

Non-null coverage:
geoid                           100.0%
det_pct                         100.0%
in_household_pct                100.0%
moved1yr_pct                    100.0%
own_pct                         100.0%
lap_pct                         100.0%
Division                        100.0%
city_centers_dist                98.9%
pop_est_5mile                    98.8%
pop_ch_1mile                     98.8%
vacant_pct                       99.1%
clip_liens_pct                   99.6%
clip_foreclosure_pct             99.6%
unq_convenience_stores_clips     99.6%
unq_gas_stations_clips           99.6%
unq_liquor_stores_clips          99.6%
transit_stop_count                3.2%
transit_stop_density              3.2%
transit_nearest_stop_m            3.2%
transit_service_intensity         3.2%
transit_overnight_stop_count      3.2%
transit_overnight_stop_share      3.2%
transit_risky_stop_count          3.2%
transit_risky_sto

In [8]:
print("Cached source pulls:")
for p in sorted((INTERIM_DIR / "sources").glob("*.parquet")):
    print(" ", p.relative_to(INTERIM_DIR.parent), f"{p.stat().st_size/1e6:.1f} MB")

feat_path = INTERIM_DIR / "features" / "bg_predictors.parquet"
print("\nFeature matrix written:", feat_path.exists(), "→", feat_path)

Cached source pulls:
  interim/sources/convenience_stores.parquet 2.7 MB
  interim/sources/demographic.parquet 10.5 MB
  interim/sources/foreclosures.parquet 2.7 MB
  interim/sources/gas_stations.parquet 2.6 MB
  interim/sources/imagery.parquet 3.6 MB
  interim/sources/liens.parquet 2.6 MB
  interim/sources/liquor_stores.parquet 2.6 MB
  interim/sources/transit.parquet 0.2 MB
  interim/sources/vacancy.parquet 3.3 MB

Feature matrix written: True → /home/eprashar_solutions_corelogic_com/crime-idx-2026/data/interim/features/bg_predictors.parquet
